### Task 1

In [1]:
from queue import PriorityQueue

class Node:
    def __init__(self, position, parent=None):
        self.position = position
        self.parent = parent
        self.g = 0
        self.h = 0
        self.f = 0

    def __lt__(self, other):
        return self.f < other.f


def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def best_first_search(maze, start, goal):
    rows, cols = len(maze), len(maze[0])
    frontier = PriorityQueue()
    start_node = Node(start)
    frontier.put(start_node)

    visited = set()

    while not frontier.empty():
        current = frontier.get()

        if current.position == goal:
            path = []
            while current:
                path.append(current.position)
                current = current.parent
            return path[::-1]

        visited.add(current.position)

        for dx, dy in [(1,0),(-1,0),(0,1),(0,-1)]:
            new_pos = (current.position[0]+dx, current.position[1]+dy)

            if (0 <= new_pos[0] < rows and
                0 <= new_pos[1] < cols and
                maze[new_pos[0]][new_pos[1]] == 0 and
                new_pos not in visited):

                new_node = Node(new_pos, current)
                new_node.h = heuristic(new_pos, goal)
                new_node.f = new_node.h
                frontier.put(new_node)

    return None


def multi_goal_search(maze, start, goals):
    current = start
    full_path = []
    goals = set(goals)

    while goals:
        nearest_goal = min(goals, key=lambda g: heuristic(current, g))

        path = best_first_search(maze, current, nearest_goal)

        if path is None:
            return None

        full_path.extend(path[1:])
        current = nearest_goal
        goals.remove(nearest_goal)

    return [start] + full_path

maze = [
[0,0,0,0],
[1,1,0,1],
[0,0,0,0],
[0,1,1,0]
]

start = (0,0)
goals = [(2,3),(3,3)]

path = multi_goal_search(maze, start, goals)

print("Full Path:", path)


Full Path: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 3), (3, 3)]


### Task 2

In [2]:
import random

graph = {
'A': {'B': 4, 'C': 3},
'B': {'D': 5, 'E': 10},
'C': {'F': 7},
'D': {'G': 6},
'E': {'G': 2},
'F': {'G': 8},
'G': {}
}

heuristic = {
'A': 10,'B': 8,'C': 5,
'D': 7,'E': 3,'F': 6,'G': 0
}


def dynamic_cost_update(graph):
    for node in graph:
        for neighbor in graph[node]:
            graph[node][neighbor] = random.randint(1,10)


def a_star_dynamic(graph, start, goal):

    frontier = [(start, heuristic[start])]
    g_cost = {start:0}
    parent = {start:None}
    visited = set()

    while frontier:

        frontier.sort(key=lambda x:x[1])
        current, f = frontier.pop(0)

        if current == goal:
            path=[]
            while current:
                path.append(current)
                current = parent[current]
            return path[::-1]

        visited.add(current)

        dynamic_cost_update(graph)

        for neighbor in graph[current]:

            new_cost = g_cost[current] + graph[current][neighbor]

            if neighbor not in g_cost or new_cost < g_cost[neighbor]:

                g_cost[neighbor] = new_cost
                f_cost = new_cost + heuristic[neighbor]

                frontier.append((neighbor,f_cost))
                parent[neighbor] = current

    return None


print("Dynamic A* Path:", a_star_dynamic(graph,'A','G'))


Dynamic A* Path: ['A', 'C', 'F', 'G']


### Task 3

In [3]:
import math

locations = {
'A': (0,0),
'B': (2,3),
'C': (5,2),
'D': (6,6),
'E': (8,3)
}

deadlines = {
'A':0,
'B':10,
'C':5,
'D':12,
'E':7
}

graph = {
'A':['B','C'],
'B':['D','E'],
'C':['E'],
'D':[],
'E':[]
}


def heuristic(a,b):
    x1,y1 = locations[a]
    x2,y2 = locations[b]
    return math.sqrt((x1-x2)**2 + (y1-y2)**2)


def greedy_delivery(graph, start):

    frontier = [(start, deadlines[start])]
    visited = set()
    parent = {start:None}

    while frontier:

        frontier.sort(key=lambda x:x[1])
        current, priority = frontier.pop(0)

        if current in visited:
            continue

        print("Visited:", current)

        visited.add(current)

        for neighbor in graph[current]:

            if neighbor not in visited:

                priority = deadlines[neighbor]
                frontier.append((neighbor,priority))
                parent[neighbor] = current

    path=[]
    node=current

    while node:
        path.append(node)
        node=parent[node]

    return path[::-1]


print("Delivery Route:", greedy_delivery(graph,'A'))


Visited: A
Visited: C
Visited: E
Visited: B
Visited: D
Delivery Route: ['A', 'B', 'D']
